# EXPERIMENT 6A – Reinforcement Learning in a Maze Environment

**Aim:** To implement Reinforcement Learning using Q-Learning in a maze environment where an agent explores the maze and learns to reach the goal.

**Algorithm:** Q-Learning (model-free reinforcement learning).

**Environment:** A small grid maze with walls, a start cell and a goal cell.

## 1. Import required libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random

# Fix random seeds so the experiment is reproducible
random.seed(42)
np.random.seed(42)

## 2. Define the maze environment

The maze is represented as a grid:
- `S` is the starting position.
- `G` is the goal.
- `#` represents a wall.
- `.` represents an empty cell.

The agent can move up, down, left or right. It receives a negative reward for each step, a penalty for hitting a wall, and a positive reward for reaching the goal.

In [ ]:
# Define the maze as a list of strings
maze = [
    "S...#...",
    ".##.#.#.",
    "...#...#",
    "#.###...",
    "...#.#.G"
]

rows = len(maze)
cols = len(maze[0])

# Find the start and goal coordinates
for r in range(rows):
    for c in range(cols):
        if maze[r][c] == "S":
            start_state = (r, c)
        if maze[r][c] == "G":
            goal_state = (r, c)

print("Start:", start_state)
print("Goal:", goal_state)
print("Maze size:", rows, "x", cols)

## 3. Define actions and rewards
Actions are numbered 0 to 3: up, down, left and right. The Q-table stores the estimated value of taking each action from each grid cell.

In [ ]:
# Action mapping: action number -> row and column movement
actions = {
    0: (-1, 0),  # Up
    1: (1, 0),   # Down
    2: (0, -1),  # Left
    3: (0, 1)    # Right
}

n_actions = len(actions)

# Create a Q-table: one row per grid cell and one column per action
Q = np.zeros((rows * cols, n_actions))

def state_to_index(state):
    # Convert (row, column) into a single Q-table row index
    return state[0] * cols + state[1]

def step(state, action):
    # Calculate the cell reached after taking the selected action
    dr, dc = actions[action]
    new_state = (state[0] + dr, state[1] + dc)

    # Hitting the maze boundary or a wall keeps the agent in place
    r, c = new_state
    if (r < 0 or r >= rows or c < 0 or c >= cols or maze[r][c] == "#"):
        return state, -5, False

    # Reaching the goal gives a large positive reward
    if new_state == goal_state:
        return new_state, 100, True

    # Every normal move has a small negative reward to encourage shorter paths
    return new_state, -1, False

## 4. Train the Q-Learning agent
Q-Learning updates its action values using the reward and the best estimated future value. Epsilon-greedy exploration sometimes chooses a random action so the agent can discover routes.

In [ ]:
# Learning parameters
learning_rate = 0.1
discount_factor = 0.95
epsilon = 1.0
epsilon_min = 0.05
epsilon_decay = 0.995
episodes = 3000
max_steps = 100

episode_rewards = []

for episode in range(episodes):
    state = start_state
    total_reward = 0

    for _ in range(max_steps):
        state_index = state_to_index(state)

        # Epsilon-greedy action selection
        if random.random() < epsilon:
            action = random.randrange(n_actions)  # Explore
        else:
            action = int(np.argmax(Q[state_index]))  # Exploit

        # Take the action and observe reward and next state
        next_state, reward, done = step(state, action)
        next_index = state_to_index(next_state)

        # Q-learning update rule
        old_value = Q[state_index, action]
        best_future_value = np.max(Q[next_index])
        Q[state_index, action] = old_value + learning_rate * (
            reward + discount_factor * best_future_value - old_value
        )

        state = next_state
        total_reward += reward

        if done:
            break

    # Gradually reduce exploration as the agent learns
    epsilon = max(epsilon_min, epsilon * epsilon_decay)
    episode_rewards.append(total_reward)

print("Training completed.")
print("Final exploration rate:", round(epsilon, 3))

## 5. Plot learning progress

In [ ]:
# Plot the average reward over blocks of 100 episodes
block_size = 100
averages = [
    np.mean(episode_rewards[i:i + block_size])
    for i in range(0, len(episode_rewards), block_size)
]

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(averages) + 1), averages, marker="o")
plt.title("Maze Q-Learning: Average Reward per 100 Episodes")
plt.xlabel("Training block (100 episodes)")
plt.ylabel("Average reward")
plt.grid(True)
plt.show()

## 6. Test the trained agent
After training, the agent chooses the action with the highest Q-value at each step. The path is displayed using coordinates.

In [ ]:
# Test the learned policy without random exploration
state = start_state
path = [state]
reached_goal = False

for _ in range(max_steps):
    action = int(np.argmax(Q[state_to_index(state)]))
    next_state, reward, done = step(state, action)

    # If the best action hits a wall, stop to avoid an unhelpful loop
    if next_state == state:
        break

    state = next_state
    path.append(state)

    if done:
        reached_goal = True
        break

print("Learned path:", path)
print("Reached goal:", reached_goal)
print("Number of moves:", len(path) - 1)

## 7. Visualize the learned path

In [ ]:
# Convert maze into a numeric image for plotting
display_grid = np.zeros((rows, cols))
for r in range(rows):
    for c in range(cols):
        if maze[r][c] == "#":
            display_grid[r, c] = 1

plt.figure(figsize=(8, 5))
plt.imshow(display_grid, cmap="Greys", origin="upper")

# Draw the path as points and connecting lines
path_rows = [p[0] for p in path]
path_cols = [p[1] for p in path]
plt.plot(path_cols, path_rows, "o-", label="Agent path")
plt.scatter(start_state[1], start_state[0], marker="s", s=150, label="Start")
plt.scatter(goal_state[1], goal_state[0], marker="*", s=250, label="Goal")
plt.xticks(range(cols))
plt.yticks(range(rows))
plt.grid(True)
plt.legend()
plt.title("Learned Maze Path")
plt.show()

## 8. Conclusion

Q-Learning was implemented in a maze environment. The agent explored the maze using an epsilon-greedy strategy, received rewards and penalties, and updated its Q-table. After training, the learned policy was tested to check whether the agent could reach the goal.

**Result:** The reward graph and the learned path are displayed when the notebook is executed.